# Signal Audit Analysis

This notebook loads an exported browser signal-audit JSON file and gives a quick diagnostic view of:
- session metadata and capabilities
- event counts by kind
- field support and whether values look constant or informative
- simple plots for motion, orientation, touch/pointer, and event timing


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)


## 1. Choose JSON file

Set `JSON_PATH` to the exported audit file you want to inspect.

In [ ]:
JSON_PATH = Path('signal_audit_d3355b30-0097-4a5d-ba1c-0d44f3d50898.json')
assert JSON_PATH.exists(), f'File not found: {JSON_PATH.resolve()}'

with JSON_PATH.open('r', encoding='utf-8') as f:
    payload = json.load(f)

events = pd.DataFrame(payload.get('events', []))
events.head()

## 2. Session summary


In [ ]:
capabilities = pd.Series(payload.get('capabilities', {}), name='value')
permissions = pd.Series(payload.get('permissions', {}), name='value')

print('startedAtIso:', payload.get('startedAtIso'))
print('auditSessionId:', payload.get('auditSessionId'))
print('n_events:', len(events))
print('event columns:', len(events.columns))

display(capabilities.to_frame('capability'))
display(permissions.to_frame('permission'))

## 3. Event counts


In [ ]:
event_counts = events['kind'].value_counts().rename_axis('kind').reset_index(name='count')
event_counts

In [ ]:
plt.figure(figsize=(10, 5))
event_counts.plot(kind='bar', x='kind', y='count', legend=False, ax=plt.gca())
plt.title('Event counts by kind')
plt.ylabel('Count')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

## 4. Field support audit

This checks selected fields for support, non-null coverage, uniqueness, and whether values appear effectively constant.

In [ ]:
FIELDS_TO_AUDIT = [
    'pointerType', 'pressure', 'width', 'height', 'force', 'radiusX', 'radiusY', 'rotationAngle',
    'ax', 'ay', 'az', 'agx', 'agy', 'agz', 'rotAlpha', 'rotBeta', 'rotGamma',
    'alpha', 'beta', 'gamma', 'scrollTop', 'valueLength'
]

def field_summary(df, field):
    if field not in df.columns:
        return {
            'field': field,
            'present': False,
            'non_null': 0,
            'null': len(df),
            'n_unique_non_null': 0,
            'numeric_min': None,
            'numeric_max': None,
            'numeric_std': None,
            'sample_values': []
        }

    s = df[field]
    non_null = s.dropna()
    numeric = pd.to_numeric(non_null, errors='coerce').dropna()

    return {
        'field': field,
        'present': True,
        'non_null': int(non_null.notna().sum()),
        'null': int(s.isna().sum()),
        'n_unique_non_null': int(non_null.astype(str).nunique()),
        'numeric_min': float(numeric.min()) if len(numeric) else None,
        'numeric_max': float(numeric.max()) if len(numeric) else None,
        'numeric_std': float(numeric.std()) if len(numeric) > 1 else None,
        'sample_values': non_null.astype(str).unique()[:8].tolist()
    }

audit_df = pd.DataFrame([field_summary(events, field) for field in FIELDS_TO_AUDIT])
audit_df

## 5. Event timing density


In [ ]:
if 'tRelMs' in events.columns:
    plt.figure(figsize=(10, 4))
    plt.hist(events['tRelMs'].dropna() / 1000.0, bins=40)
    plt.title('Event timing density')
    plt.xlabel('Time since session start (s)')
    plt.ylabel('Event count')
    plt.tight_layout()
    plt.show()
else:
    print('tRelMs not present')

## 6. Motion plots


In [ ]:
motion = events[events['kind'] == 'devicemotion'].copy()
motion[['tRelMs', 'ax', 'ay', 'az', 'agx', 'agy', 'agz', 'rotAlpha', 'rotBeta', 'rotGamma']].head()

In [ ]:
if not motion.empty:
    t = motion['tRelMs'] / 1000.0

    plt.figure(figsize=(10, 4))
    plt.plot(t, motion['ax'], label='ax')
    plt.plot(t, motion['ay'], label='ay')
    plt.plot(t, motion['az'], label='az')
    plt.title('Acceleration (without gravity)')
    plt.xlabel('Time (s)')
    plt.ylabel('m/s²-ish browser units')
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(t, motion['agx'], label='agx')
    plt.plot(t, motion['agy'], label='agy')
    plt.plot(t, motion['agz'], label='agz')
    plt.title('Acceleration including gravity')
    plt.xlabel('Time (s)')
    plt.ylabel('m/s²-ish browser units')
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4))
    plt.plot(t, motion['rotAlpha'], label='rotAlpha')
    plt.plot(t, motion['rotBeta'], label='rotBeta')
    plt.plot(t, motion['rotGamma'], label='rotGamma')
    plt.title('Rotation rate')
    plt.xlabel('Time (s)')
    plt.ylabel('Rotation rate')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No devicemotion events found')

## 7. Orientation plots


In [ ]:
orientation = events[events['kind'] == 'deviceorientation'].copy()
orientation[['tRelMs', 'alpha', 'beta', 'gamma']].head()

In [ ]:
if not orientation.empty:
    t = orientation['tRelMs'] / 1000.0
    plt.figure(figsize=(10, 4))
    plt.plot(t, orientation['alpha'], label='alpha')
    plt.plot(t, orientation['beta'], label='beta')
    plt.plot(t, orientation['gamma'], label='gamma')
    plt.title('Device orientation')
    plt.xlabel('Time (s)')
    plt.ylabel('Degrees')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No deviceorientation events found')

## 8. Touch / pointer contact fields


In [ ]:
touch_pointer = events[events['kind'].isin(['pointerdown', 'pointermove', 'pointerup', 'touchstart', 'touchmove', 'touchend'])].copy()
touch_pointer[['kind', 'pointerType', 'pressure', 'width', 'height', 'force', 'radiusX', 'radiusY', 'rotationAngle']].head(20)

In [ ]:
for field in ['width', 'height', 'pressure', 'force', 'radiusX', 'radiusY', 'rotationAngle']:
    if field in touch_pointer.columns and touch_pointer[field].notna().any():
        plt.figure(figsize=(8, 4))
        plt.hist(pd.to_numeric(touch_pointer[field], errors='coerce').dropna(), bins=30)
        plt.title(f'Distribution of {field}')
        plt.xlabel(field)
        plt.ylabel('Count')
        plt.tight_layout()
        plt.show()
    else:
        print(f'{field}: no usable values found')

## 9. Quick conclusions helper


In [ ]:
def interpret_field(row):
    if not row['present']:
        return 'missing'
    if row['non_null'] == 0:
        return 'all null'
    if row['n_unique_non_null'] <= 1:
        return 'constant or near-constant'
    if row['numeric_std'] is not None and row['numeric_std'] == 0:
        return 'constant numeric'
    return 'varies'

quick_conclusions = audit_df[['field']].copy()
quick_conclusions['interpretation'] = audit_df.apply(interpret_field, axis=1)
quick_conclusions